# Predicting next-month arrears (`arrears_next_month`)
Protocol: `protocol.yaml`. Temporal holdout from `holdout.csv`; 5-fold walk-forward CV on training rows; choose by mean CV ROC AUC; score the chosen model once on the holdout.

**Feature decisions** (from `columns.csv`): `meter_ref` and `billing_month` are excluded per protocol. `final_notice_sent` is also excluded: it is recorded in the billing cycle *after* the billing month, i.e. after prediction time, and it is nearly a proxy for the outcome (87% arrears when 1 vs 1.4% when 0). Using it would be leakage.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data.csv')
hold = pd.read_csv('holdout.csv')
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
hold_idx = hold['row'].to_numpy()
assert len(df)==20939 and len(hold_idx)==7536
is_hold = np.zeros(len(df), bool); is_hold[hold_idx]=True
train = df[~is_hold]; test = df[is_hold]
print(len(train), len(test), train.billing_month.max(), test.billing_month.min())

In [ ]:
TARGET='arrears_next_month'
FEATURES=['tariff','occupants','kwh','bill_eur','late_payments_12m']
CAT=['tariff']; NUM=[f for f in FEATURES if f not in CAT]
def pre(scale=True):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
# All candidates named before any scoring
candidates = {
 'baseline_prior': lambda: Pipeline([('p',pre()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':    lambda: Pipeline([('p',pre()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':      lambda: Pipeline([('p',pre()),('m',LogisticRegression(C=1,max_iter=1000))]),
 'rf_depth6':      lambda: Pipeline([('p',pre(False)),('m',RandomForestClassifier(300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'gbm_depth2':     lambda: Pipeline([('p',pre(False)),('m',GradientBoostingClassifier(n_estimators=150,max_depth=2,learning_rate=0.05,random_state=0))]),
 'gbm_depth3':     lambda: Pipeline([('p',pre(False)),('m',GradientBoostingClassifier(n_estimators=150,max_depth=3,learning_rate=0.05,random_state=0))]),
}
print(list(candidates))

In [ ]:
# Walk-forward CV by billing month inside training rows: 5 folds, each validates on a block of later months
months = np.sort(train.billing_month.unique())
blocks = np.array_split(months, 6)   # first block = initial train only
folds=[]
for k in range(1,6):
    tr_m = np.concatenate(blocks[:k]); va_m = blocks[k]
    tr = np.where(train.billing_month.isin(tr_m))[0]; va = np.where(train.billing_month.isin(va_m))[0]
    folds.append((tr,va))
    print(k, tr_m[0], tr_m[-1], '->', va_m[0], va_m[-1], len(tr), len(va))
y = train[TARGET].to_numpy()
cv_scores = {}
for name,mk in candidates.items():
    s=[]
    for tr,va in folds:
        m=mk().fit(train.iloc[tr][FEATURES], y[tr])   # preprocessing fitted inside the fold
        s.append(roc_auc_score(y[va], m.predict_proba(train.iloc[va][FEATURES])[:,1]))
    cv_scores[name]=s
cvdf=pd.DataFrame(cv_scores).T; cvdf['mean']=cvdf.mean(axis=1); cvdf.sort_values('mean',ascending=False)

In [ ]:
chosen = cvdf['mean'].idxmax(); print('chosen by mean CV roc_auc:', chosen)
final = candidates[chosen]().fit(train[FEATURES], y)
holdout_auc = roc_auc_score(test[TARGET], final.predict_proba(test[FEATURES])[:,1])   # scored once
print('holdout roc_auc:', holdout_auc)

In [ ]:
train_pos = np.where(~is_hold)[0]
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[train_pos[tr].tolist(), train_pos[va].tolist()] for tr,va in folds],
 'cv_scores': {k:[float(x) for x in v] for k,v in cv_scores.items()},
 'chosen': chosen,
 'holdout_rows': hold_idx.tolist(),
 'holdout_scores': {chosen: float(holdout_auc)},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. Predictions use only features known at the billing month; `final_notice_sent` was deliberately left out as post-outcome information.